# 🛡️ ARMOR: Adaptive Relearning-resistant Multimodal Unlearning

> **Kaggle GPU Notebook** — Runs on T4 GPU (free tier, 30 hrs/week)

## What this notebook does
1. **Setup** — Clones the ARMOR repo and installs dependencies (GPU-enabled)
2. **Smoke Tests** — Verifies all 10 methods pass in debug mode
3. **Full Experiments** — Runs GA, NPO, NPO+SAM, RMU on Mistral-7B with TOFU
4. **Relearning Attack** — Tests resistance of unlearned models to adversarial recovery
5. **Visualization** — Plots forget quality vs retain accuracy for all methods

---
### 🗂️ Experiment Menu
| Cell | What it runs | GPU time |
|------|-------------|----------|
| §2 | Smoke tests (debug/CPU) | ~3 min |
| §3 | GA baseline (Mistral-7B) | ~20 min |
| §4 | NPO baseline (Mistral-7B) | ~25 min |
| §5 | NPO+SAM — ARMOR core (Mistral-7B) | ~40 min |
| §6 | RMU (Mistral-7B) | ~30 min |
| §7 | Relearning Attack comparison | ~20 min |
| §8 | Results visualization | <1 min |

**Total budget: ~2.5 hrs GPU time** (well within 30 hr/week free quota)

---
⚙️ **Before running:** Enable GPU in *Settings → Accelerator → GPU T4 x2*

## § 0 — Environment Check

In [ ]:
import os, sys, subprocess, platform, json
from datetime import datetime

# ── GPU check ────────────────────────────────────────────────────────────────
try:
    import torch
    cuda_ok   = torch.cuda.is_available()
    gpu_name  = torch.cuda.get_device_name(0) if cuda_ok else "None"
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9 if cuda_ok else 0
    print(f"🟢 CUDA available : {cuda_ok}")
    print(f"   GPU            : {gpu_name}")
    print(f"   VRAM           : {vram_gb:.1f} GB")
    print(f"   PyTorch        : {torch.__version__}")
except ImportError:
    print("⚠️  PyTorch not yet installed — will install in §1")
    cuda_ok = False

print(f"\n🐍 Python    : {sys.version.split()[0]}")
print(f"🖥️  Platform  : {platform.platform()}")
print(f"📅 Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if not cuda_ok:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   → Go to Kaggle Settings → Accelerator → GPU T4 x2")
    print("   → Then restart the kernel and re-run this cell")
else:
    print(f"\n✅ Ready to run full ARMOR experiments on GPU!")

## § 1 — Install Dependencies & Clone Repo

In [ ]:
# ── Clone ARMOR from GitHub ───────────────────────────────────────────────────
REPO_URL  = "https://github.com/Angrajkarn/-ARMOR-Adaptive-Relearning-resistant-Multimodal-Unlearning"
REPO_DIR  = "/kaggle/working/armor_project"

import os, subprocess

if os.path.exists(REPO_DIR):
    print(f"📂 Repo already cloned at {REPO_DIR} — pulling latest...")
    result = subprocess.run(["git", "pull"], cwd=REPO_DIR, capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print(f"📥 Cloning ARMOR repo to {REPO_DIR}...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ Cloned successfully!")
    else:
        print(f"❌ Clone failed:\n{result.stderr}")
        raise RuntimeError("Git clone failed")

# ── Add repo to Python path ───────────────────────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n📁 Working directory: {os.getcwd()}")
print("📦 Project files:")
for f in sorted(os.listdir('.')):
    print(f"   {f}")

In [ ]:
# ── Install Python dependencies ───────────────────────────────────────────────
print("📦 Installing ARMOR dependencies...\n")

# Core packages (bitsandbytes enabled for GPU QLoRA)
deps = [
    "torch>=2.1.0",
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",   # QLoRA — GPU only, essential for Mistral-7B
    "rouge-score>=0.1.2",
    "scikit-learn>=1.4.0",
    "numpy>=1.26.0",
    "tqdm>=4.66.0",
    "rich>=13.7.0",
    "Pillow>=10.0.0",         # LLaVA multimodal
    "torchvision>=0.16.0",    # LLaVA image preprocessing
    "matplotlib>=3.8.0",      # Results plotting
    "pandas>=2.0.0",          # Results tables
    "seaborn>=0.13.0",        # Heatmaps
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + deps,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ All dependencies installed successfully")
else:
    print("⚠️  Some warnings (usually OK):")
    print(result.stderr[-2000:])   # last 2000 chars only

# Verify critical imports
import importlib
critical = ["torch", "transformers", "datasets", "peft", "bitsandbytes"]
print("\n🔍 Verifying critical imports:")
for pkg in critical:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "?")
        print(f"   ✅ {pkg:<20} {ver}")
    except ImportError as e:
        print(f"   ❌ {pkg:<20} MISSING — {e}")

In [ ]:
# ── Hugging Face Token Setup ──────────────────────────────────────────────────
# Required for: LLaMA-2 (gated model), TOFU dataset download
# Optional for: Mistral-7B (public, no token needed)
#
# How to get your HF Token:
#   1. Go to https://huggingface.co/settings/tokens
#   2. Click 'New token' → name it 'kaggle-armor' → Role: 'Read'
#   3. Copy the token and paste it in the Kaggle Secret below:
#      Kaggle → Add-ons → Secrets → Add → Name: HF_TOKEN

from kaggle_secrets import UserSecretsClient
import os

HF_TOKEN = None
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("✅ HF Token loaded from Kaggle Secrets")
    print(f"   Token preview: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
except Exception as e:
    print(f"⚠️  No HF_TOKEN secret found: {e}")
    print("   → Mistral-7B will still work (public model)")
    print("   → LLaMA-2 requires a token (gated access)")
    print("\n   To add token: Kaggle → Add-ons → Secrets → Add HF_TOKEN")

# Choose experiment model based on token availability
USE_MISTRAL = True       # Mistral-7B — no token needed, 7B params
USE_LLAMA2  = HF_TOKEN is not None  # LLaMA-2 — requires token + gated access
USE_QLORA   = True       # 4-bit quantization — essential for 7B on 16GB GPU

print(f"\n🔧 Experiment Configuration:")
print(f"   USE_MISTRAL : {USE_MISTRAL}  (Mistral-7B-v0.1)")
print(f"   USE_LLAMA2  : {USE_LLAMA2}  (Llama-2-7b-hf)")
print(f"   USE_QLORA   : {USE_QLORA}   (4-bit QLoRA — saves ~10GB VRAM)")

## § 2 — Smoke Tests (All 10 Methods, Debug Mode)

In [ ]:
# ── Run all smoke tests in debug mode ────────────────────────────────────────
# This uses distilgpt2 (tiny model) to verify all pipelines work on Kaggle
# before committing GPU time to full experiments

print("🧪 Running ARMOR Smoke Test Suite (debug mode, distilgpt2)...")
print("   Expected: 10/10 PASS in ~3 minutes\n")

result = subprocess.run(
    [sys.executable, "scripts/run_smoke_tests.py"],
    cwd=REPO_DIR,
    capture_output=False,
    text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)

if result.returncode == 0:
    print("\n✅ All smoke tests PASSED — safe to run full experiments!")
else:
    print(f"\n❌ Some tests FAILED (exit code {result.returncode})")
    print("   Fix failures before running full experiments")

## § 3 — Full Experiment: Gradient Ascent (GA) Baseline

In [ ]:
# ── Gradient Ascent Baseline ──────────────────────────────────────────────────
# GA is the simplest unlearning baseline: maximize forget-set loss
# Known weakness: creates sharp minima → easily reversed by relearning attacks

print("🚀 Running GA Baseline on Mistral-7B with QLoRA...")
print("   Expected time: ~20 minutes on T4 GPU")
print("   forget10 split → 10% of TOFU authors forgotten\n")

ga_cmd = [
    sys.executable, "scripts/run_baseline_ga.py",
    "--model", "mistral-7b",
    "--qlora",
    "--output-dir", "outputs/ga",
]
if HF_TOKEN:
    ga_cmd += ["--hf-token", HF_TOKEN]

print(f"CMD: {' '.join(ga_cmd)}\n")

result_ga = subprocess.run(
    ga_cmd, cwd=REPO_DIR, text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)
GA_SUCCESS = result_ga.returncode == 0
print(f"\n{'✅ GA DONE' if GA_SUCCESS else '❌ GA FAILED'} (exit code {result_ga.returncode})")

## § 4 — Full Experiment: NPO Baseline

In [ ]:
# ── Negative Preference Optimization (NPO) ────────────────────────────────────
# NPO uses a DPO-style log-ratio loss for more stable unlearning than GA
# Still vulnerable to relearning attacks (creates sharp minima)

print("🚀 Running NPO Baseline on Mistral-7B with QLoRA...")
print("   Expected time: ~25 minutes on T4 GPU\n")

npo_cmd = [
    sys.executable, "scripts/run_baseline_npo.py",
    "--model", "mistral-7b",
    "--qlora",
    "--output-dir", "outputs/npo",
]
if HF_TOKEN:
    npo_cmd += ["--hf-token", HF_TOKEN]

print(f"CMD: {' '.join(npo_cmd)}\n")

result_npo = subprocess.run(
    npo_cmd, cwd=REPO_DIR, text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)
NPO_SUCCESS = result_npo.returncode == 0
print(f"\n{'✅ NPO DONE' if NPO_SUCCESS else '❌ NPO FAILED'} (exit code {result_npo.returncode})")

## § 5 — Full Experiment: NPO + SAM (ARMOR Core Method)

In [ ]:
# ── NPO + SAM — ARMOR's relearning-resistant core ────────────────────────────
# SAM (Sharpness-Aware Minimization) finds FLAT minima in the loss landscape
# → even if an attacker fine-tunes on forget data, the model stays "forgotten"
#
# SAM adds a 2-phase update per step (perturb → update at perturbed point)
# → ~2× compute per epoch, but much stronger robustness

print("🛡️ Running NPO+SAM (ARMOR Core) on Mistral-7B with QLoRA...")
print("   Expected time: ~40 minutes on T4 GPU")
print("   SAM ρ=0.05 (default neighbourhood radius)\n")

npo_sam_cmd = [
    sys.executable, "scripts/run_npo_sam.py",
    "--model", "mistral-7b",
    "--qlora",
    "--sam-rho", "0.05",
    "--run-mia",               # Run Membership Inference Audit
    "--output-dir", "outputs/npo_sam",
]
if HF_TOKEN:
    npo_sam_cmd += ["--hf-token", HF_TOKEN]

print(f"CMD: {' '.join(npo_sam_cmd)}\n")

result_npo_sam = subprocess.run(
    npo_sam_cmd, cwd=REPO_DIR, text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)
NPO_SAM_SUCCESS = result_npo_sam.returncode == 0
print(f"\n{'✅ NPO+SAM DONE' if NPO_SAM_SUCCESS else '❌ NPO+SAM FAILED'} (exit code {result_npo_sam.returncode})")

## § 6 — Full Experiment: RMU (Representation Misdirection Unlearning)

In [ ]:
# ── RMU — Representation Misdirection Unlearning ─────────────────────────────
# Instead of manipulating output logits (like GA/NPO), RMU corrupts the
# INTERNAL REPRESENTATIONS at layer ~70% depth.
#
# Loss: α·‖h_L(x_f) − c·u‖² + β·‖h_L(x_r) − h_L_ref(x_r)‖²
#   x_f = forget input, x_r = retain input, u = random unit vector
#   This pushes forget representations toward noise → harder to reverse

print("🔀 Running RMU on Mistral-7B with QLoRA...")
print("   Expected time: ~30 minutes on T4 GPU")
print("   α=1200 (forget misdirection weight), β=6.5 (retain preservation)\n")

rmu_cmd = [
    sys.executable, "scripts/run_rmu.py",
    "--model", "mistral-7b",
    "--qlora",
    "--alpha", "1200",
    "--beta",  "6.5",
    "--run-mia",
    "--output-dir", "outputs/rmu",
]
if HF_TOKEN:
    rmu_cmd += ["--hf-token", HF_TOKEN]

print(f"CMD: {' '.join(rmu_cmd)}\n")

result_rmu = subprocess.run(
    rmu_cmd, cwd=REPO_DIR, text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)
RMU_SUCCESS = result_rmu.returncode == 0
print(f"\n{'✅ RMU DONE' if RMU_SUCCESS else '❌ RMU FAILED'} (exit code {result_rmu.returncode})")

## § 7 — Relearning Attack: Can Attackers Recover Forgotten Knowledge?

In [ ]:
# ── Relearning Attack ─────────────────────────────────────────────────────────
# Simulates an adversary who fine-tunes the unlearned model on forget samples
# to try to recover the forgotten knowledge.
#
# ARMOR's hypothesis: SAM-based unlearning creates flat minima that are
# resistant to this attack, unlike GA/NPO which create sharp minima.
#
# Measures: accuracy recovery % after N epochs of fine-tuning

print("⚔️  Running Relearning Attack on all 3 unlearned checkpoints...")
print("   GA checkpoint    : outputs/ga/ga_unlearned")
print("   NPO checkpoint   : outputs/npo/npo_unlearned")
print("   NPO+SAM checkpoint: outputs/npo_sam/npo_sam_unlearned")
print("   Expected time: ~20 minutes total\n")

attack_cmd = [
    sys.executable, "scripts/run_relearning_attack.py",
    "--model", "mistral-7b",
    "--qlora",
    "--compare",
    "--ga-checkpoint",   "outputs/ga/ga_unlearned",
    "--npo-checkpoint",  "outputs/npo/npo_unlearned",
    "--sam-checkpoint",  "outputs/npo_sam/npo_sam_unlearned",
    "--epochs", "10",
]
if HF_TOKEN:
    attack_cmd += ["--hf-token", HF_TOKEN]

print(f"CMD: {' '.join(attack_cmd)}\n")

result_attack = subprocess.run(
    attack_cmd, cwd=REPO_DIR, text=True,
    env={**os.environ, "PYTHONIOENCODING": "utf-8"}
)
ATTACK_SUCCESS = result_attack.returncode == 0
print(f"\n{'✅ Attack analysis DONE' if ATTACK_SUCCESS else '❌ Attack FAILED'} (exit code {result_attack.returncode})")

## § 8 — Results Visualization

In [ ]:
# ── Visualization: Forget Quality vs Retain Accuracy ─────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ─── Data: fill in from your experiment outputs ──────────────────────────────
# Format: (method_name, forget_quality, retain_accuracy, mia_auroc)
# forget_quality: higher = better forgetting (closer to 1.0 = fully forgotten)
# retain_accuracy: higher = model still performs well on non-forgotten data
# mia_auroc: closer to 0.5 = harder for attacker to distinguish members

RESULTS = [
    # Method         FQ     RA      MIA-AUROC
    ("Pre-unlearn",  0.00,  0.85,   0.95),   # Before any unlearning
    ("GA",           0.72,  0.63,   0.61),   # Gradient Ascent — hurts retain
    ("NPO",          0.80,  0.76,   0.55),   # Better retain, still sharp
    ("NPO+SAM",      0.88,  0.81,   0.51),   # ARMOR — flat minima, balanced
    ("RMU",          0.85,  0.78,   0.52),   # Representation-level forgetting
]
# NOTE: Replace the values above with actual numbers from §3-§6 outputs!

methods = [r[0] for r in RESULTS]
fq      = [r[1] for r in RESULTS]
ra      = [r[2] for r in RESULTS]
mia     = [r[3] for r in RESULTS]

colors  = ["#6B7280", "#EF4444", "#F59E0B", "#10B981", "#6366F1"]
markers = ["X",       "o",       "s",       "*",       "D"]
sizes   = [120,       120,       120,       250,       150]

# ── Figure 1: Forget Quality vs Retain Accuracy scatter ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor("#0F172A")

for ax in axes:
    ax.set_facecolor("#1E293B")
    ax.tick_params(colors="#94A3B8", labelsize=10)
    ax.spines[:].set_color("#334155")

# --- Plot 1: FQ vs RA ---
ax1 = axes[0]
for i, (m, f, r, _) in enumerate(RESULTS):
    ax1.scatter(f, r, c=colors[i], marker=markers[i], s=sizes[i],
                zorder=5, label=m, edgecolors="white", linewidths=0.5)
    ax1.annotate(m, (f, r), textcoords="offset points",
                 xytext=(8, 4), fontsize=8, color="#CBD5E1")

# Ideal zone annotation
ax1.axhline(y=0.80, color="#10B981", linestyle="--", alpha=0.4, linewidth=1)
ax1.axvline(x=0.80, color="#10B981", linestyle="--", alpha=0.4, linewidth=1)
ax1.text(0.82, 0.82, "Ideal Zone", color="#10B981", fontsize=9, alpha=0.7)

ax1.set_xlabel("Forget Quality ↑", color="#94A3B8", fontsize=11)
ax1.set_ylabel("Retain Accuracy ↑", color="#94A3B8", fontsize=11)
ax1.set_title("Forget Quality vs Retain Accuracy", color="#F1F5F9", fontsize=13, pad=15)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(0.4, 1.0)
ax1.grid(True, alpha=0.15, color="#334155")

# --- Plot 2: MIA AUROC bar chart ---
ax2 = axes[1]
bars = ax2.barh(methods, mia, color=colors, alpha=0.85, height=0.6)
ax2.axvline(x=0.5, color="#10B981", linestyle="--", alpha=0.6, linewidth=2,
            label="Ideal (0.5 = random)")

for bar, val in zip(bars, mia):
    ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", color="#F1F5F9", fontsize=10)

ax2.set_xlabel("MIA AUROC (lower = better privacy) ↓", color="#94A3B8", fontsize=11)
ax2.set_title("Membership Inference Attack AUROC", color="#F1F5F9", fontsize=13, pad=15)
ax2.set_xlim(0.4, 1.05)
ax2.tick_params(axis='y', colors="#CBD5E1")
ax2.legend(facecolor="#1E293B", labelcolor="#94A3B8", fontsize=9)
ax2.grid(True, alpha=0.15, color="#334155", axis="x")

# --- Plot 3: Composite score bar ---
ax3 = axes[2]
# Composite = 0.4*FQ + 0.4*RA + 0.2*(1-|MIA-0.5|*2)
composite = [0.4*f + 0.4*r + 0.2*(1 - abs(m - 0.5)*2) for f, r, m in zip(fq, ra, mia)]
bars3 = ax3.bar(methods, composite, color=colors, alpha=0.85, width=0.6)

for bar, val in zip(bars3, composite):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f"{val:.3f}", ha="center", color="#F1F5F9", fontsize=10)

ax3.set_ylabel("Composite Score ↑", color="#94A3B8", fontsize=11)
ax3.set_title("Overall ARMOR Score\n(0.4×FQ + 0.4×RA + 0.2×Privacy)",
              color="#F1F5F9", fontsize=12, pad=15)
ax3.set_ylim(0, 1.05)
ax3.tick_params(axis='x', rotation=15, colors="#CBD5E1")
ax3.grid(True, alpha=0.15, color="#334155", axis="y")

plt.suptitle("ARMOR Unlearning — Method Comparison on TOFU (Mistral-7B)",
             color="#F1F5F9", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("outputs/armor_results.png", dpi=150, bbox_inches="tight",
            facecolor="#0F172A")
plt.show()
print("\n📊 Plot saved to outputs/armor_results.png")

In [ ]:
# ── Results Summary Table ─────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(RESULTS, columns=["Method", "Forget Quality", "Retain Accuracy", "MIA AUROC"])
df["Composite"] = [0.4*f + 0.4*r + 0.2*(1 - abs(m - 0.5)*2)
                   for f, r, m in zip(df["Forget Quality"], df["Retain Accuracy"], df["MIA AUROC"])]
df = df.sort_values("Composite", ascending=False).reset_index(drop=True)

# Highlight best values
def highlight_best(s):
    if s.name in ["Forget Quality", "Retain Accuracy", "Composite"]:
        is_max = s == s.max()
        return ["background-color: #064e3b; color: #6ee7b7" if v else "" for v in is_max]
    elif s.name == "MIA AUROC":
        # For MIA AUROC, closest to 0.5 is best
        is_best = abs(s - 0.5) == abs(s - 0.5).min()
        return ["background-color: #064e3b; color: #6ee7b7" if v else "" for v in is_best]
    return ["" for _ in s]

print("\n📊 ARMOR Method Comparison — TOFU Benchmark (Mistral-7B + QLoRA)")
print("   Green = best value in each column\n")
print("   NOTE: Replace placeholder values with actual experiment outputs!\n")

styled = (df.style
           .apply(highlight_best)
           .format({"Forget Quality": "{:.4f}",
                   "Retain Accuracy": "{:.4f}",
                   "MIA AUROC": "{:.4f}",
                   "Composite": "{:.4f}"})
           .set_caption("ARMOR Unlearning Results — Higher is Better (except MIA AUROC: closer to 0.5)"))
display(styled)

## § 9 — Advanced: SAM Hyperparameter Search

In [ ]:
# ── Grid search over SAM ρ (neighbourhood radius) ─────────────────────────────
# ρ controls the size of the worst-case perturbation SAM considers
# Larger ρ → flatter minima (more robust) but slower convergence
# Typical range: 0.01 to 0.2

# ⚠️  This cell takes ~2 hours — only run if you have remaining GPU quota
RUN_HP_SEARCH = False  # Set to True to enable

if RUN_HP_SEARCH:
    import json

    rho_values = [0.01, 0.02, 0.05, 0.10, 0.20]
    hp_results = []

    for rho in rho_values:
        print(f"\n{'='*50}")
        print(f"  Testing SAM ρ = {rho}")
        print(f"{'='*50}")

        cmd = [
            sys.executable, "scripts/run_npo_sam.py",
            "--model", "mistral-7b",
            "--qlora",
            "--sam-rho", str(rho),
            "--no-rouge",
            "--output-dir", f"outputs/hp_search/rho_{rho}",
        ]
        if HF_TOKEN:
            cmd += ["--hf-token", HF_TOKEN]

        r = subprocess.run(cmd, cwd=REPO_DIR, text=True,
                          env={**os.environ, "PYTHONIOENCODING": "utf-8"})
        hp_results.append({"rho": rho, "success": r.returncode == 0})
        print(f"ρ={rho}: {'✅ Done' if r.returncode == 0 else '❌ Failed'}")

    print("\n🔍 HP Search Complete:")
    for entry in hp_results:
        print(f"  ρ={entry['rho']}: {entry['success']}")
else:
    print("⏭️  HP search skipped (set RUN_HP_SEARCH=True to enable)")
    print("   Recommended ρ = 0.05 from ARMOR paper")

## § 10 — Advanced: RMU Layer Sensitivity Analysis

In [ ]:
# ── RMU: Which layer to misdirect? ───────────────────────────────────────────
# RMU hooks into a transformer layer to corrupt hidden states.
# Default: 70% depth (e.g., layer 22/32 for Mistral-7B)
# This analysis tests different layer choices to find the optimal target

# ⚠️  Each run is ~30 min — total ~2 hours
RUN_LAYER_ANALYSIS = False  # Set to True to enable

if RUN_LAYER_ANALYSIS:
    # For Mistral-7B (32 layers), test at 40%, 55%, 70%, 85% depth
    layer_depths = [(40, 13), (55, 18), (70, 22), (85, 27)]

    for pct, layer_idx in layer_depths:
        print(f"\n{'='*50}")
        print(f"  RMU at layer {layer_idx}/32 (~{pct}% depth)")
        print(f"{'='*50}")

        cmd = [
            sys.executable, "scripts/run_rmu.py",
            "--model", "mistral-7b",
            "--qlora",
            "--layer", str(layer_idx),
            "--no-rouge",
            "--output-dir", f"outputs/layer_study/layer_{layer_idx}",
        ]
        if HF_TOKEN:
            cmd += ["--hf-token", HF_TOKEN]

        r = subprocess.run(cmd, cwd=REPO_DIR, text=True,
                          env={**os.environ, "PYTHONIOENCODING": "utf-8"})
        print(f"Layer {layer_idx}: {'✅' if r.returncode == 0 else '❌'}")
else:
    print("⏭️  Layer analysis skipped (set RUN_LAYER_ANALYSIS=True to enable)")
    print("   Default: 70% depth works well across most architectures")

## § 11 — Save Results & Download

In [ ]:
# ── Package results for download ──────────────────────────────────────────────
import zipfile, glob, shutil

OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
ZIP_PATH    = "/kaggle/working/armor_results.zip"

print("📦 Packaging experiment results...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Include logs and plots (but NOT model weights — too large)
    for pattern in ["**/*.json", "**/*.png", "**/*.csv", "**/*.txt", "**/*.log"]:
        for fpath in glob.glob(os.path.join(OUTPUTS_DIR, pattern), recursive=True):
            arcname = os.path.relpath(fpath, REPO_DIR)
            zf.write(fpath, arcname)
            print(f"   + {arcname}")

zip_size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n✅ Zipped to: {ZIP_PATH}")
print(f"   Size: {zip_size_mb:.1f} MB")
print(f"   Download via: Kaggle → Output → Download")

# Print final summary
print("\n" + "="*60)
print("  ARMOR KAGGLE RUN — FINAL SUMMARY")
print("="*60)
summary = [
    ("Smoke Tests", True),
    ("GA Baseline", GA_SUCCESS),
    ("NPO Baseline", NPO_SUCCESS),
    ("NPO+SAM (ARMOR)", NPO_SAM_SUCCESS),
    ("RMU", RMU_SUCCESS),
    ("Relearning Attack", ATTACK_SUCCESS),
]
for name, ok in summary:
    icon = "✅" if ok else "❌"
    print(f"  {icon} {name}")
print("="*60)

---

## 📋 Quick Reference

### Running individual experiments
```bash
# Quick debug (CPU, 2 min)
python scripts/run_npo_sam.py --debug --no-rouge

# Full GPU experiment (Mistral-7B + QLoRA)
python scripts/run_npo_sam.py --model mistral-7b --qlora --run-mia

# RMU with custom layer
python scripts/run_rmu.py --model mistral-7b --qlora --layer 18 --alpha 1200

# Relearning attack on saved checkpoint
python scripts/run_relearning_attack.py --checkpoint outputs/npo_sam/npo_sam_unlearned --compare
```

### Key metrics
| Metric | Ideal | Meaning |
|--------|-------|---------|
| Forget Quality | → 1.0 | Model has forgotten the target data |
| Retain Accuracy | → 1.0 | Model still works on non-forgotten data |
| MIA AUROC | → 0.5 | Attacker cannot distinguish members (privacy) |
| Acc. Recovery % | → 0% | Attack cannot recover forgotten knowledge |

### Kaggle GPU budget
- **Free**: 30 hrs T4 GPU/week
- **This notebook**: ~2.5 hrs total
- **Remaining**: ~27.5 hrs for additional experiments

### HF Token setup
1. [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → New token → Read
2. Kaggle → Add-ons → Secrets → Add → Name: `HF_TOKEN`
3. Re-run §0 to load the token